Copyright 2018-2026 AVEVA Group Limited

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

   http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
SPDX-License-Identifier: Apache-2.0

##Step 1: Install the CONNECT data services sample python library
Install the [AVEVA CONNECT data services python sample library](https://github.com/AVEVA/sample-adh-sample_libraries-python).\


In [0]:
%python
# Install a library using %pip. Only need to run once.
%pip install adh_sample_library_preview\
# Restart the Python process
dbutils.library.restartPython()

##Step 2: Create client-credentials secret scope
Store client-credentials client in secret stope or set them explicitly. 

####Option 1 - Create a secret scope using Databricks CLI:
This option requires the ability to use the Databricks CLI in the workspace terminal or in command prompt.
1. In CONNECT data services, create a client-credentials client and temporarily save the client id and secret generated. For this sample, the client-credentials client needs to be given a role that has read and write access to a CONNECT data services namespace. For instructions, see:\
[Add a client-credentials client](https://docs.aveva.com/bundle/connect-data-services/page/1263324.html)
2. Create a secret scope called `cdsscope` For instructions using the Databricks CLI, see:\
[Databricks secret management](https://docs.databricks.com/en/security/secrets/index.html)
3. Create two secrets within that secret scope called `cdsclientid` and `cdsclientsecret` containing the client id and secret generated from CONNECT data services. Refer to the link in step 2 for instructions.

####Option 2 - Create a secret scope using Databricks SDK for Python:
_Note: This uses the Databricks SDK for Python option which requires entering your client-credentials into the notebook in plain text. It's recommended to delete these credentials after the block is run._
1. In CONNECT data services, create a client-credentials client and temporarily save the client id and secret generated. For this sample, the client-credentials client needs to be given a role that has read and write access to a CONNECT data services namespace. For instructions, see:\
[Add a client-credentials client](https://docs.aveva.com/bundle/connect-data-services/page/1263324.html)
2. Enter your client credentials into the option 2 code block
3. Run the option 2 code block.

In [0]:
# Use this to skip setting scope if already done
scopeAlreadyCreated = False

if scopeAlreadyCreated == False:
    
    from databricks.sdk import WorkspaceClient

    # Enter your client id and secret here
    clientId = "YOUR_CLIENT_ID"
    clientSecret = "YOUR_CLIENT_SECRET"

    w = WorkspaceClient()

    # Create scope
    try: 
        w.secrets.create_scope(scope='cdsscope')
    except Exception as e:
        print(e)

    # Create secrets
    try:
        w.secrets.put_secret(scope='cdsscope', key='cdsclientid', string_value=clientId)
        w.secrets.put_secret(scope='cdsscope', key='cdsclientsecret', string_value=clientSecret)
    except Exception as e:
        print(e)

##Step 3: Import libraries and create client connection
Import libraries, set other connection information, and create the CONNECT data services client connection.

In [0]:
import json
import random
import requests
import time
from adh_sample_library_preview import ADHClient, SdsType, SdsTypeCode, SdsTypeProperty, SdsStream
from datetime import datetime

# Retrieve secrets from Databricks secrets
clientId = dbutils.secrets.get(scope = "cdsscope", key = "cdsclientid")
clientSecret = dbutils.secrets.get(scope = "cdsscope", key = "cdsclientsecret")
# Set connection information
apiVersion = "v1"
resource = "https://uswe.datahub.connect.aveva.com" #change if using a region other than US West
tenantId = "YOUR_TENANT_ID"
namespaceId = "YOUR_NAMESPACE_ID"

# Set Sds type and stream name
typeName = "ValueTimestampType"
streamName = "databricks_stream_random"

# Create connection
cds_client = ADHClient(
            apiVersion,
            tenantId,
            resource,
            clientId,
            clientSecret
        )

##Step 4: Create the Sequential Data Store (SDS) type
Create the SDS type if it does not exist. If it does, get the definition.

An SDS type defines the shape and structure of events and how to associate events within a stream of data. A type is comprised of at least two properties. One property serves as the primary index, most commonly a timestamp or DateTime. In addition, it has one or more additional properties called value properties that describe the data in each stream event. Each value property can have a different property type. A wide variety of property types are supported.

For more information see: [CONNECT data services developer documentation - types](https://docs.aveva.com/bundle/connect-data-services-developer/page/developer-guide/sequential-data-store-dev/sds-types-dev.html)

In [0]:
def get_type_value_time():
    double_type = SdsType("doubleType", SdsTypeCode.Double)
    time_type = SdsType("string", SdsTypeCode.DateTime)
    value = SdsTypeProperty("Value", False, double_type)
    time_prop = SdsTypeProperty("Timestamp", True, time_type)

    type_value_time = SdsType(typeName, SdsTypeCode.Object, [value, time_prop],
                              description="A Time-Series indexed type with a value")

    return type_value_time

try: 
    time_value_type = get_type_value_time()
    time_value_type = cds_client.Types.getOrCreateType(namespaceId, time_value_type)
except Exception as e:
    print(e)

##Step 5: Create or get the SDS stream
Create the SDS stream if it does not exist. If it does, get the definition.

In [0]:
stream = SdsStream(streamName, time_value_type.Id,
                                    description="A stream for created by the Sample Databricks to CONNECT Notebook")
try:
    cds_client.Streams.createOrUpdateStream(namespaceId, stream)
except Exception as e:
    print(e)

##Step 6: Write data to the stream using the streams REST API
Write random data to the stream. By default, this will write one value. Change `loop = True` if you would like to continuously update the data every minute. If you need to write lots of data in bulk, using Open Message Format (OMF) is a better choice. See steps 6 and later for an example. 

In [0]:
# Set to True to run until cancelled
loop = False
# Set how often you want to send data (in seconds)
repeat = 10

# Modify this with your data source
def get_data():
    values = []
    values.append({"Value": random.random(), "Timestamp": datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%S.%fZ")})
    return values

while True:
    try:
        cds_client.Streams.insertValues(namespaceId,stream.Id, json.dumps((get_data())))
        valueWritten = cds_client.Streams.getLastValue(namespaceId,stream.Id)
        print(valueWritten)
    except Exception as e:
        print(e)
    if loop == False:
        break
    time.sleep(repeat)


##Step 7: Tests
Run this block after running the above blocks to test.

In [0]:
import unittest
from unittest.mock import patch, MagicMock

class TestNotebook(unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        global namespaceId, typeName, streamName, cds_client
        cls.namespaceId = namespaceId
        cls.typeName = typeName
        cls.streamName = streamName
        cls.cds_client = cds_client

    def test_get_type_value_time(self):
        typeName = "ValueTimestampType"
        def get_type_value_time():
            double_type = SdsType("doubleType", SdsTypeCode.Double)
            time_type = SdsType("string", SdsTypeCode.DateTime)
            value = SdsTypeProperty("Value", False, double_type)
            time_prop = SdsTypeProperty("Timestamp", True, time_type)
            type_value_time = SdsType(typeName, SdsTypeCode.Object, [value, time_prop],
                                      description="A Time-Series indexed type with a value")
            return type_value_time
        result = get_type_value_time()
        self.assertEqual(result.Id, typeName)
        self.assertEqual(result.SdsTypeCode, SdsTypeCode.Object)
        self.assertEqual(len(result.Properties), 2)

    def test_get_data(self):
        from datetime import datetime
        def get_data():
            values = []
            values.append({"Value": 0.5, "Timestamp": datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%S.%fZ")})
            return values
        data = get_data()
        self.assertIsInstance(data, list)
        self.assertIn("Value", data[0])
        self.assertIn("Timestamp", data[0])

    @patch('adh_sample_library_preview.ADHClient')
    def test_stream_insert_and_get_last_value(self, MockADHClient):
        mock_client = MockADHClient()
        mock_stream = MagicMock()
        mock_stream.Id = "stream_id"
        mock_client.Streams.insertValues.return_value = None
        mock_client.Streams.getLastValue.return_value = {"Value": 1.0, "Timestamp": "2025-12-29T12:00:00.000Z"}
        namespaceId = "namespace"
        values = [{"Value": 1.0, "Timestamp": "2025-12-29T12:00:00.000Z"}]
        import json
        mock_client.Streams.insertValues(namespaceId, mock_stream.Id, json.dumps(values))
        result = mock_client.Streams.getLastValue(namespaceId, mock_stream.Id)
        self.assertEqual(result["Value"], 1.0)
        self.assertIn("Timestamp", result)

    def test_type_exists(self):
        sds_type = self.cds_client.Types.getType(self.namespaceId, self.typeName)
        self.assertIsNotNone(sds_type)
        self.assertEqual(sds_type.Id, self.typeName)
        self.assertEqual(sds_type.SdsTypeCode, SdsTypeCode.Object)
        self.assertTrue(any(p.Id == "Value" for p in sds_type.Properties))
        self.assertTrue(any(p.Id == "Timestamp" for p in sds_type.Properties))

    def test_stream_exists(self):
        stream = self.cds_client.Streams.getStream(self.namespaceId, self.streamName)
        self.assertIsNotNone(stream)
        self.assertEqual(stream.Id, self.streamName)
        self.assertEqual(stream.TypeId, self.typeName)

    def test_data_written(self):
        last_value = self.cds_client.Streams.getLastValue(self.namespaceId, self.streamName)
        self.assertIsNotNone(last_value)
        self.assertIn("Value", last_value)
        self.assertIn("Timestamp", last_value)

# Note: If this cell is run multiple times in a notebook, the class may be redefined,
# causing unittest to register duplicate test methods. Restart the Python process or
# clear the notebook state to avoid duplicate test discovery.

if __name__ == '__main__':
    unittest.main(argv=['first-arg-is-ignored'], exit=False)